# 06 — Jaccard Range Overlap (Full Scale)

Computes Jaccard range overlap across all 6,466 plant species and
24,939 pollinator species. Used as a reference analysis — in the full
ANTHEIA pipeline, spatial co-occurrence is encoded via the existence
matrices F and P and the shared bin count N, not by Jaccard directly.

## Why Jaccard, not raw co-occurrence counts?

Observation density in GBIF and PhenoField tracks human population density
via iNaturalist. A raw co-occurrence count or density product systematically
favors species pairs that are both common and observed in high-density areas
(urban and suburban regions near major cities), regardless of their ecological
relationship.

The Jaccard index normalizes by the union of both species' ranges:

```
Jaccard(plant, pollinator) = |bins_plant ∩ bins_pollinator| / |bins_plant ∪ bins_pollinator|
```

A species with a broad range observed across many bins will have a large
denominator, reducing its Jaccard score with any given partner unless
they genuinely co-occur across most of that range. This controls for the
observation bias that inflates co-occurrence counts for common species
in well-sampled areas.

This correction was introduced after the initial density-score analysis
showed top-ranked bins clustering systematically around urban areas rather
than reflecting genuine ecological co-occurrence patterns.

**Note:** At full scale (6,466 × 24,939 pairs), the full pairwise
computation is expensive. This notebook uses the same min-heap approach
as the small-scale version to extract the top 100 pairs without
materializing the full matrix.

In [ ]:
import pandas as pd
import numpy as np
import heapq
from pathlib import Path
import gc

BASE     = Path("/scratch/ariana.l")
F_PATH   = BASE / "Stage 4 Link Prediction Model" / "stage4_F_existence_phenofield.csv"
P_PATH   = BASE / "New Stage 4 Link Prediction Model" / "stage4_P_existence_corrected.csv"
OUT_DIR  = BASE / "New Stage 4 Link Prediction Model"

TOP_N    = 100

print("Paths OK")

In [ ]:
# Load F and P matrices (already restricted to common bins)
print("Loading F matrix...")
F_df = pd.read_csv(F_PATH, index_col=0)
print(f"  F shape: {F_df.shape}")

print("Loading P matrix...")
P_df = pd.read_csv(P_PATH, index_col=0)
print(f"  P shape: {P_df.shape}")

In [ ]:
# Build per-species bin sets from binary matrices
# For Jaccard: bin set = set of columns where value = 1
print("Building bin sets from F matrix...")
plant_bin_sets = {}
for sp in F_df.index:
    row = F_df.loc[sp]
    plant_bin_sets[sp] = set(row[row == 1].index)

print("Building bin sets from P matrix...")
pollinator_bin_sets = {}
for sp in P_df.index:
    row = P_df.loc[sp]
    pollinator_bin_sets[sp] = set(row[row == 1].index)

print(f"Plant species: {len(plant_bin_sets):,}")
print(f"Pollinator species: {len(pollinator_bin_sets):,}")

del F_df, P_df
gc.collect()

In [ ]:
# Compute Jaccard for all pairs, keep top 100
# Jaccard(A, B) = |A ∩ B| / |A ∪ B|
# Normalizes by union — controls for range size and observation density bias

print("Computing Jaccard range overlap...")
jaccard_heap = []

plant_list = list(plant_bin_sets.items())

for i, (p_species, p_bins) in enumerate(plant_list):
    for q_species, q_bins in pollinator_bin_sets.items():
        intersection = p_bins & q_bins
        if not intersection:
            continue
        jaccard = len(intersection) / len(p_bins | q_bins)
        entry = (jaccard, len(intersection), p_species, q_species)
        if len(jaccard_heap) < TOP_N:
            heapq.heappush(jaccard_heap, entry)
        elif jaccard > jaccard_heap[0][0]:
            heapq.heapreplace(jaccard_heap, entry)

    if (i + 1) % 500 == 0:
        print(f"  plant species {i+1:,} / {len(plant_list):,}")

print(f"Done. Top {len(jaccard_heap)} pairs found.")

In [ ]:
# Convert to DataFrame and save
jaccard_results = pd.DataFrame(
    jaccard_heap,
    columns=['jaccard', 'shared_bins', 'plant_species', 'pollinator_species']
).sort_values('jaccard', ascending=False).reset_index(drop=True)

jaccard_results.insert(0, 'rank', jaccard_results.index + 1)
jaccard_results['jaccard'] = jaccard_results['jaccard'].round(4)

out_path = OUT_DIR / 'top100_jaccard_pairs_fullscale.csv'
jaccard_results.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(jaccard_results.head(20).to_string(index=False))